# Conformal Prediction: 90% Coverage, 29% Where It Counts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/conformal_prediction.ipynb)

Split conformal prediction turns any model's output into a set or an interval that
contains the truth at a rate you choose. The guarantee needs no assumption about the
model, the noise or the data distribution, only that the calibration data and the test
data are exchangeable.

This notebook builds the method, verifies the guarantee over 200 random splits, then
measures the two places it does less than it appears to: coverage conditional on where
you look, and coverage after exchangeability is broken.

Companion post: [sesen.ai/blog/conformal-prediction-python](https://sesen.ai/blog/conformal-prediction-python)

Runtime: about two minutes on a CPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, fetch_covtype
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.preprocessing import StandardScaler

ALPHA = 0.10          # 1 - ALPHA is the coverage we are asking for

## 1. The method

Score every calibration point, take a quantile of those scores, use it as a threshold.
The ceiling adjustment is not a rounding detail: `np.quantile` would lose roughly `1/n`
of coverage, which is invisible at n = 2000 and fatal at n = 100. The `n + 1` counts the
test point itself, because the argument is about the rank of a new score among `n + 1`.

In [ ]:
def conformal_quantile(scores, alpha):
    """The ceiling-adjusted (1 - alpha) quantile of the calibration scores."""
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    if k > n:
        return np.inf
    return float(np.sort(scores)[k - 1])


# A 90% interval is impossible with fewer than 9 calibration points.
print(conformal_quantile(np.arange(8.0), 0.10))
print(conformal_quantile(np.arange(9.0), 0.10))

## 2. The guarantee, verified

California housing, three disjoint splits: the model trains on the first, the conformal
quantile is computed on the second, and every coverage number comes off the third. The
score is the absolute residual.

Any one split lands anywhere between 87% and 92%, because the test set carries its own
sampling noise. Averaged over splits, coverage must land inside
`[1 - alpha, 1 - alpha + 1/(n_cal + 1)]`.

In [ ]:
X, y = fetch_california_housing(return_X_y=True)


def split_conformal(seed, n_train=4000, n_cal=2000, n_test=4000):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))[: n_train + n_cal + n_test]
    tr, ca, te = idx[:n_train], idx[n_train : n_train + n_cal], idx[n_train + n_cal :]

    model = RandomForestRegressor(n_estimators=60, random_state=seed, n_jobs=-1).fit(X[tr], y[tr])
    qhat = conformal_quantile(np.abs(y[ca] - model.predict(X[ca])), ALPHA)

    covered = np.abs(y[te] - model.predict(X[te])) <= qhat
    return covered.mean(), 2 * qhat


coverage, width = zip(*[split_conformal(s) for s in range(200)])
coverage = np.array(coverage)

print(f"mean coverage {coverage.mean():.5f}   requested {1 - ALPHA:.5f}")
print(f"guaranteed band [{1 - ALPHA:.5f}, {1 - ALPHA + 1 / 2001:.5f}]")
print(f"single splits ranged {coverage.min():.4f} to {coverage.max():.4f}")
print(f"mean interval width {np.mean(width):.3f}  (units of $100,000)")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))

a1.hist(coverage, bins=22, color="#bfe4e4", edgecolor="#1f9e9e")
a1.axvline(0.90, color="#d99120", lw=2, label="requested 90.0%")
a1.axvline(coverage.mean(), color="#4a6d8c", lw=2, ls="--", label=f"mean {coverage.mean():.3%}")
a1.set_xlabel("empirical coverage")
a1.set_ylabel("splits")
a1.legend(frameon=False)

a2.plot(np.arange(1, 201), np.cumsum(coverage) / np.arange(1, 201), color="#1f9e9e")
a2.axhline(0.90, color="#d99120")
a2.fill_between([1, 200], 0.90, 0.90 + 1 / 2001, color="#d99120", alpha=0.25)
a2.set_ylim(0.8955, 0.9045)
a2.set_xlabel("splits averaged")
a2.set_ylabel("running mean coverage")
plt.tight_layout()
plt.show()

## 3. The coverage level is a dial

Nothing about the model changes between these rows. Only the quantile moves.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(y))[:10000]
tr, ca, te = idx[:4000], idx[4000:6000], idx[6000:]
model = RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1).fit(X[tr], y[tr])
s_cal = np.abs(y[ca] - model.predict(X[ca]))
s_test = np.abs(y[te] - model.predict(X[te]))

for a in [0.50, 0.25, 0.10, 0.05, 0.01]:
    q = conformal_quantile(s_cal, a)
    print(f"requested {1 - a:5.0%}   achieved {(s_test <= q).mean():6.1%}   width {2 * q:5.2f}")

## 4. The guarantee is an average

Marginal coverage is averaged over the joint distribution of x and y. It says nothing
about any particular region. Here is one feature, one target, and noise that grows
twelvefold from left to right.

In [ ]:
def heteroscedastic_1d(n, seed=1):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 10, n)
    mu = 2.0 * np.sin(1.1 * x) + 0.4 * x
    sd = 0.15 + 0.165 * x
    return x[:, None], mu + rng.normal(0, sd, n), mu, sd


X1, y1, mu1, sd1 = heteroscedastic_1d(14000, seed=1)
perm = np.random.default_rng(7).permutation(len(y1))
tr, ca, te = perm[:3000], perm[3000:6000], perm[6000:]

base = GradientBoostingRegressor(random_state=0).fit(X1[tr], y1[tr])
q_const = conformal_quantile(np.abs(y1[ca] - base.predict(X1[ca])), ALPHA)

centre = base.predict(X1[te])
inside = np.abs(y1[te] - centre) <= q_const
xt = X1[te, 0]
edges = np.quantile(xt, np.linspace(0, 1, 6))

print(f"marginal coverage {inside.mean():.1%}   constant width {2 * q_const:.2f}")
for i in range(5):
    m = (xt >= edges[i]) & (xt <= edges[i + 1])
    print(f"  x in [{edges[i]:4.1f}, {edges[i+1]:4.1f}]   coverage {inside[m].mean():5.1%}")

## 5. Two repairs

Both keep the conformal machinery untouched and change the score function, so that a
score of `qhat` means something different at different x.

**Normalised residuals** divide by a predicted error magnitude. The scale model is fitted
on training data only, so the score function is fixed before calibration begins.

**Conformalised quantile regression** skips the mean entirely and lets conformal
prediction correct a pair of quantile fits. Its score is signed, so an over-wide quantile
band gets a negative `qhat` and is shrunk rather than padded.

In [ ]:
# normalised residual
resid_tr = np.abs(y1[tr] - base.predict(X1[tr]))
scale_model = GradientBoostingRegressor(random_state=0).fit(X1[tr], np.log(resid_tr + 1e-6))
scale = lambda Z: np.exp(scale_model.predict(Z)) + 1e-6
q_norm = conformal_quantile(np.abs(y1[ca] - base.predict(X1[ca])) / scale(X1[ca]), ALPHA)

# conformalised quantile regression
lo_model = GradientBoostingRegressor(loss="quantile", alpha=ALPHA / 2, random_state=0).fit(X1[tr], y1[tr])
hi_model = GradientBoostingRegressor(loss="quantile", alpha=1 - ALPHA / 2, random_state=0).fit(X1[tr], y1[tr])


def cqr_score(y, lo, hi):
    """How far outside the quantile band y fell. Negative if comfortably inside."""
    return np.maximum(lo - y, y - hi)


q_cqr = conformal_quantile(cqr_score(y1[ca], lo_model.predict(X1[ca]), hi_model.predict(X1[ca])), ALPHA)


def bands(Z):
    c = base.predict(Z)
    return {
        "constant":   (c - q_const, c + q_const),
        "normalised": (c - q_norm * scale(Z), c + q_norm * scale(Z)),
        "CQR":        (lo_model.predict(Z) - q_cqr, hi_model.predict(Z) + q_cqr),
    }


header = f"{'':<12}{'marginal':>10}" + "".join(f"{f'{edges[i]:.0f}-{edges[i+1]:.0f}':>9}" for i in range(5)) + f"{'width':>8}"
print(header)
for name, (lo, hi) in bands(X1[te]).items():
    ins = (y1[te] >= lo) & (y1[te] <= hi)
    per = "".join(f"{ins[(xt >= edges[i]) & (xt <= edges[i+1])].mean():>9.1%}" for i in range(5))
    print(f"{name:<12}{ins.mean():>10.1%}{per}{(hi - lo).mean():>8.2f}")

In [ ]:
grid = np.linspace(0, 10, 400)[:, None]
gb = bands(grid)
sub = np.random.default_rng(0).choice(te, 900, replace=False)
z90 = 1.6448536269514722
g = grid.ravel()
truth_mu, truth_sd = 2.0 * np.sin(1.1 * g) + 0.4 * g, 0.15 + 0.165 * g

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, key in zip(axes, ["constant", "normalised", "CQR"]):
    lo, hi = gb[key]
    ax.scatter(X1[sub, 0], y1[sub], s=3, color="#8a8a8a", alpha=0.3, linewidths=0)
    ax.fill_between(g, lo, hi, color="#1f9e9e", alpha=0.3)
    ax.plot(g, truth_mu - z90 * truth_sd, color="#d99120", ls="--")
    ax.plot(g, truth_mu + z90 * truth_sd, color="#d99120", ls="--", label="true 90% band")
    ax.set_title(key)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

## 6. Prediction sets

For classification the output is a set of labels. Forest cover type, seven classes.

**LAC** (least ambiguous set-valued classifier) keeps every label above a fixed
probability threshold. **APS** (adaptive prediction sets) accumulates probability mass
instead, so the set grows on rows where the model spread its mass thin.

Watch the marginal number, then watch the per-class column.

In [ ]:
COVER = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow",
         "Aspen", "Douglas-fir", "Krummholz"]

XC, yC = fetch_covtype(return_X_y=True)
yC = yC - 1
ic = np.random.default_rng(0).permutation(len(yC))[:26000]
trc, cac, tec = ic[:12000], ic[12000:18000], ic[18000:]

sc = StandardScaler().fit(XC[trc])
clf = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1).fit(sc.transform(XC[trc]), yC[trc])
P_cal, P_te = clf.predict_proba(sc.transform(XC[cac])), clf.predict_proba(sc.transform(XC[tec]))
y_cal, y_te = yC[cac], yC[tec]
rows = np.arange(len(y_te))

print(f"top-1 accuracy {(P_te.argmax(1) == y_te).mean():.1%}")

In [ ]:
q_lac = conformal_quantile(1 - P_cal[np.arange(len(y_cal)), y_cal], ALPHA)
S_lac = P_te >= 1 - q_lac

print(f"LAC   marginal coverage {S_lac[rows, y_te].mean():.2%}   mean set size {S_lac.sum(1).mean():.2f}")
print(f"\n{'class':<20}{'n':>6}{'accuracy':>10}{'coverage':>10}{'set size':>10}")
for k in np.argsort([-(S_lac[y_te == j][:, j].mean()) for j in range(7)]):
    m = y_te == k
    print(f"{COVER[k]:<20}{m.sum():>6}{(P_te[m].argmax(1) == k).mean():>10.1%}"
          f"{S_lac[m][:, k].mean():>10.1%}{S_lac[m].sum(1).mean():>10.2f}")

The marginal 90% is real, and it was paid for by the two classes holding most of the
rows. Adaptive prediction sets spend size where the model is unsure, which lifts the hard
classes without touching the guarantee.

In [ ]:
def aps_scores(probs, labels, u):
    """Cumulative mass down to the true label, minus a random slice of its own mass."""
    order = np.argsort(-probs, axis=1)
    srt = np.take_along_axis(probs, order, axis=1)
    cum = np.cumsum(srt, axis=1)
    rank = np.argmax(order == labels[:, None], axis=1)
    r = np.arange(len(labels))
    return cum[r, rank] - u * srt[r, rank]


def aps_sets(probs, qhat, u):
    """Peel labels off in descending order until the cumulative mass passes qhat."""
    order = np.argsort(-probs, axis=1)
    srt = np.take_along_axis(probs, order, axis=1)
    cum = np.cumsum(srt, axis=1)
    keep = (cum - u[:, None] * srt) <= qhat
    keep[:, 0] = True                       # never return an empty set
    sets = np.zeros_like(keep)
    np.put_along_axis(sets, order, keep, axis=1)
    return sets


u_cal = np.random.default_rng(1).uniform(size=len(y_cal))
u_te = np.random.default_rng(2).uniform(size=len(y_te))
q_aps = conformal_quantile(aps_scores(P_cal, y_cal, u_cal), ALPHA)
S_aps = aps_sets(P_te, q_aps, u_te)

print(f"APS   marginal coverage {S_aps[rows, y_te].mean():.2%}   mean set size {S_aps.sum(1).mean():.2f}")
print(f"\n{'class':<20}{'LAC cov':>10}{'APS cov':>10}{'APS size':>10}")
for k in [1, 2, 0, 6, 5, 3, 4]:
    m = y_te == k
    print(f"{COVER[k]:<20}{S_lac[m][:, k].mean():>10.1%}{S_aps[m][:, k].mean():>10.1%}{S_aps[m].sum(1).mean():>10.2f}")

### Set size is a difficulty score

The size of the set is a per-row signal on a scale you chose, needing no calibration step
of its own. Routing every row with a set larger than one to a human is a triage rule you
can state in a sentence.

In [ ]:
size = S_aps.sum(1)
correct = P_te.argmax(1) == y_te
print(f"{'set size':>10}{'rows':>8}{'top-1 accuracy':>18}")
for s in range(1, 8):
    m = size == s
    if m.sum():
        print(f"{s:>10}{m.sum():>8}{correct[m].mean():>18.1%}")

## 7. Breaking exchangeability

Exchangeability is the one assumption, so break it and measure the damage. California
housing splits at latitude 36: the north holds the Bay Area and the Central Valley, the
south holds Los Angeles and San Diego. Train and calibrate on the north, deploy on the
south.

Nothing in the method notices. No warning, no widening interval, no diagnostic.

In [ ]:
housing = fetch_california_housing()
FEAT = list(housing.feature_names)
lat = X[:, FEAT.index("Latitude")]
north = lat >= 36

rng = np.random.default_rng(0)
ins, outs = np.flatnonzero(north), np.flatnonzero(~north)
rng.shuffle(ins)
rng.shuffle(outs)
tr_h, ca_h, held_h, te_h = ins[:4000], ins[4000:6000], ins[6000:10000], outs[:6000]

mh = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X[tr_h], y[tr_h])
qh = conformal_quantile(np.abs(y[ca_h] - mh.predict(X[ca_h])), ALPHA)

print(f"interval width {2 * qh:.2f} in both cases")
print(f"held out, same region    {(np.abs(y[held_h] - mh.predict(X[held_h])) <= qh).mean():.1%}")
print(f"deployed, southern half  {(np.abs(y[te_h] - mh.predict(X[te_h])) <= qh).mean():.1%}")

In [ ]:
frac, cov = [], []
for f in np.linspace(0, 1, 11):
    n_out = int(round(f * 4000))
    sel = np.concatenate([held_h[: 4000 - n_out], te_h[:n_out]])
    frac.append(f)
    cov.append((np.abs(y[sel] - mh.predict(X[sel])) <= qh).mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.scatter(X[ca_h[:1500], FEAT.index("Longitude")], X[ca_h[:1500], FEAT.index("Latitude")],
           s=4, color="#1f9e9e", alpha=0.5, linewidths=0, label="calibrated here")
a1.scatter(X[te_h[:1500], FEAT.index("Longitude")], X[te_h[:1500], FEAT.index("Latitude")],
           s=4, color="#d99120", alpha=0.5, linewidths=0, label="deployed here")
a1.axhline(36, color="#2b2b2b", ls="--", lw=1)
a1.set_xlabel("longitude")
a1.set_ylabel("latitude")
a1.legend(frameon=False, markerscale=2.5)

a2.plot(frac, cov, color="#d99120", marker="o")
a2.axhline(0.90, color="#2b2b2b", ls="--", lw=1)
a2.set_xlabel("fraction of the test set drawn from the south")
a2.set_ylabel("coverage")
plt.tight_layout()
plt.show()

## Exercises

1. **Break the ceiling.** Replace `conformal_quantile` with `np.quantile(scores, 1 - alpha)`
   and rerun section 2 with `n_cal` set to 100, 300 and 1000. Plot the shortfall against
   `1/n_cal` and confirm it matches the theory.

2. **Class-conditional conformal.** Compute a separate `qhat` per class on the covtype data
   (Mondrian conformal prediction) and check what it does to Aspen's coverage and to the
   mean set size. How few calibration rows can a class have before its own quantile becomes
   useless?

3. **Weighted conformal under shift.** Fit a logistic regression to distinguish northern
   from southern rows, turn its output into a likelihood ratio, and use those as weights in
   the calibration quantile. How much of the drop from 89.8% to 37.2% does it recover?

4. **A different score.** For regression, try the signed score `y - f(x)` with two separate
   quantiles at `alpha/2` and `1 - alpha/2` instead of the absolute residual. When does an
   asymmetric interval beat a symmetric one?

5. **Time series.** Generate an autoregressive series, calibrate on the first half and test
   on the second, and measure how far coverage falls. Then try calibrating on a random
   permutation of the whole series and explain why that number is not trustworthy either.

## Where this comes from

- Vovk, Gammerman and Shafer (2005), *Algorithmic Learning in a Random World*
- Papadopoulos, Proedrou, Vovk and Gammerman (2002), "Inductive Confidence Machines for Regression"
- Lei, G'Sell, Rinaldo, Tibshirani and Wasserman (2018), "Distribution-Free Predictive Inference for Regression"
- Romano, Patterson and Candès (2019), "Conformalized Quantile Regression"
- Tibshirani, Barber, Candès and Ramdas (2019), "Conformal Prediction Under Covariate Shift"
- Angelopoulos and Bates (2021), "A Gentle Introduction to Conformal Prediction"
- Vovk (2012), "Conditional Validity of Inductive Conformal Predictors"